In [1]:
import pandas as pd
#Extracting the data set from github
url = "https://raw.githubusercontent.com/KeithGalli/Pandas-Data-Science-Tasks/master/SalesAnalysis/Output/all_data.csv"
df = pd.read_csv(url)

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Dataset loaded successfully!
Rows: 186850
Columns: 6


In [2]:
df.head()#Returns the first 5 rows of the DataFrame
df.info()#Prints a concise summary of the DataFrame
df.isnull().sum()#Checks the entire DataFrame for missing values

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 186850 entries, 0 to 186849
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   Order ID          186305 non-null  object
 1   Product           186305 non-null  object
 2   Quantity Ordered  186305 non-null  object
 3   Price Each        186305 non-null  object
 4   Order Date        186305 non-null  object
 5   Purchase Address  186305 non-null  object
dtypes: object(6)
memory usage: 8.6+ MB


,0
Order ID,545
Product,545
Quantity Ordered,545
Price Each,545
Order Date,545
Purchase Address,545


In [3]:
# performing Data cleaning now
# Check duplicate rows
print("Duplicate rows before cleaning:", df.duplicated().sum())

# Remove completely empty rows
df = df.dropna(how="all")

# Remove repeated header rows
df = df[df["Order Date"] != "Order Date"]

# Convert numeric columns
df["Quantity Ordered"] = pd.to_numeric(df["Quantity Ordered"])
df["Price Each"] = pd.to_numeric(df["Price Each"])

# Convert date column
df["Order Date"] = pd.to_datetime(df["Order Date"])

# Remove exact duplicate rows
df = df.drop_duplicates()

# Check the cleaned dataset
print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Rows after cleaning:", len(df))

df.info()

Duplicate rows before cleaning: 1162


/tmp/ipykernel_3454/125871151.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Order Date"] = pd.to_datetime(df["Order Date"])


Duplicate rows after cleaning: 0
Rows after cleaning: 185686
<class 'pandas.core.frame.DataFrame'>
Index: 185686 entries, 0 to 186849
Data columns (total 6 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   Order ID          185686 non-null  object        
 1   Product           185686 non-null  object        
 2   Quantity Ordered  185686 non-null  int64         
 3   Price Each        185686 non-null  float64       
 4   Order Date        185686 non-null  datetime64[ns]
 5   Purchase Address  185686 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(3)
memory usage: 9.9+ MB


In [4]:
# Create Sales column
df["Sales"] = df["Quantity Ordered"] * df["Price Each"]

# Create Month column
df["Month"] = df["Order Date"].dt.month

# Extract City from Purchase Address
df["City"] = df["Purchase Address"].str.split(",").str[1].str.strip()

# Extract State from Purchase Address
df["State"] = df["Purchase Address"].str.split(",").str[2].str.strip().str[:2]

# Check the new columns
print(df.columns.tolist())

# Preview the new features
df[["Order Date", "Sales", "Month", "City", "State"]].head()

['Order ID', 'Product', 'Quantity Ordered', 'Price Each', 'Order Date', 'Purchase Address', 'Sales', 'Month', 'City', 'State']


,Order Date,Sales,Month,City,State
0,2019-04-19 08:46:00,23.90,4,Dallas,TX
2,2019-04-07 22:30:00,99.99,4,Boston,MA
3,2019-04-12 14:38:00,600.00,4,Los Angeles,CA
4,2019-04-12 14:38:00,11.99,4,Los Angeles,CA
5,2019-04-30 09:27:00,11.99,4,Los Angeles,CA


In [5]:
# Final validation

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

# Create SQLite database
import sqlite3

conn = sqlite3.connect("ecommerce_sales.db")

print("\nSQLite database connected successfully!")

# Load DataFrame into SQLite
df.to_sql(
    "sales_data",
    conn,
    if_exists="replace",
    index=False
)

print("Data loaded into SQLite successfully!")

# Test SQL table
query = """
SELECT *
FROM sales_data
LIMIT 5;
"""

result = pd.read_sql_query(query, conn)

result

Rows: 185686
Columns: 10

Missing values:
Order ID            0
Product             0
Quantity Ordered    0
Price Each          0
Order Date          0
Purchase Address    0
Sales               0
Month               0
City                0
State               0
dtype: int64

Data types:
Order ID                    object
Product                     object
Quantity Ordered             int64
Price Each                 float64
Order Date          datetime64[ns]
Purchase Address            object
Sales                      float64
Month                        int32
City                        object
State                       object
dtype: object

SQLite database connected successfully!
Data loaded into SQLite successfully!


,Order ID,Product,Quantity Ordered,Price Each,Order Date,Purchase Address,Sales,Month,City,State
0,176558,USB-C Charging Cable,2,11.95,2019-04-19 08:46:00,"917 1st St, Dallas, TX 75001",23.90,4,Dallas,TX
1,176559,Bose SoundSport Headphones,1,99.99,2019-04-07 22:30:00,"682 Chestnut St, Boston, MA 02215",99.99,4,Boston,MA
2,176560,Google Phone,1,600.00,2019-04-12 14:38:00,"669 Spruce St, Los Angeles, CA 90001",600.00,4,Los Angeles,CA
3,176560,Wired Headphones,1,11.99,2019-04-12 14:38:00,"669 Spruce St, Los Angeles, CA 90001",11.99,4,Los Angeles,CA
4,176561,Wired Headphones,1,11.99,2019-04-30 09:27:00,"333 8th St, Los Angeles, CA 90001",11.99,4,Los Angeles,CA


In [6]:
#Now sql business analysis
query = """
SELECT ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data;
"""

result = pd.read_sql_query(query, conn)

result

,Total_Revenue
0,34465537.94


In [7]:
#Total Quantity sold
query = """
SELECT SUM("Quantity Ordered") AS Total_Quantity_Sold
FROM sales_data;
"""

result = pd.read_sql_query(query, conn)

result

,Total_Quantity_Sold
0,208812


In [8]:
#Total orders
query = """
SELECT COUNT(DISTINCT "Order ID") AS Total_Orders
FROM sales_data;
"""

result = pd.read_sql_query(query, conn)

result

,Total_Orders
0,178437


In [9]:
#revenue by month
query = """
SELECT
    Month,
    ROUND(SUM(Sales), 2) AS Monthly_Revenue
FROM sales_data
GROUP BY Month
ORDER BY Month;
"""

result = pd.read_sql_query(query, conn)

result

,Month,Monthly_Revenue
0,1,1821413.16
1,2,2200078.08
2,3,2804973.35
3,4,3389217.98
4,5,3150616.23
5,6,2576280.15
6,7,2646461.32
7,8,2241083.37
8,9,2094465.69
9,10,3734777.86


In [10]:
#revenue by product
query = """
SELECT
    Product,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY Product
ORDER BY Total_Revenue DESC;
"""

result = pd.read_sql_query(query, conn)

result.head(10)

,Product,Total_Revenue
0,Macbook Pro Laptop,8032500.00
1,iPhone,4792900.00
2,ThinkPad Laptop,4127958.72
3,Google Phone,3317400.00
4,27in 4K Gaming Monitor,2433147.61
5,34in Ultrawide Monitor,2352898.08
6,Apple Airpods Headphones,2345550.00
7,Flatscreen TV,1443900.00
8,Bose SoundSport Headphones,1342865.70
9,27in FHD Monitor,1131074.59


In [11]:
#quantity sold by product
query = """
SELECT
    Product,
    SUM("Quantity Ordered") AS Total_Quantity
FROM sales_data
GROUP BY Product
ORDER BY Total_Quantity DESC;
"""

result = pd.read_sql_query(query, conn)

result.head(10)

,Product,Total_Quantity
0,AAA Batteries (4-pack),30986
1,AA Batteries (4-pack),27615
2,USB-C Charging Cable,23931
3,Lightning Charging Cable,23169
4,Wired Headphones,20524
5,Apple Airpods Headphones,15637
6,Bose SoundSport Headphones,13430
7,27in FHD Monitor,7541
8,iPhone,6847
9,27in 4K Gaming Monitor,6239


In [12]:
#  Calculate total revenue generated by each city

query = """
SELECT
    City,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY City
ORDER BY Total_Revenue DESC;
"""

# Run the SQL query and store the result in a DataFrame
result = pd.read_sql_query(query, conn)

# Display the top 10 cities by revenue
result.head(10)

,City,Total_Revenue
0,San Francisco,8254743.55
1,Los Angeles,5448304.28
2,New York City,4661867.14
3,Boston,3658627.65
4,Atlanta,2794199.07
5,Dallas,2765373.96
6,Seattle,2745046.02
7,Portland,2319331.94
8,Austin,1818044.33


In [13]:
# Calculate total revenue generated by each state

query = """
SELECT
    State,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY State
ORDER BY Total_Revenue DESC;
"""

# Execute the SQL query
result = pd.read_sql_query(query, conn)

# Display all states and their revenue
result

,State,Total_Revenue
0,CA,13703047.83
1,NY,4661867.14
2,TX,4583418.29
3,MA,3658627.65
4,GA,2794199.07
5,WA,2745046.02
6,OR,1870010.56
7,ME,449321.38


In [14]:
# Calculate the Average Order Value (AOV)
# AOV = Total Revenue / Total Number of Orders

query = """
SELECT
    ROUND(
        SUM(Sales) / COUNT(DISTINCT "Order ID"),
        2
    ) AS Average_Order_Value
FROM sales_data;
"""

# Execute the SQL query
result = pd.read_sql_query(query, conn)

# Display the Average Order Value
result

,Average_Order_Value
0,193.15


In [15]:
#  Calculate the number of unique orders received each month

query = """
SELECT
    Month,
    COUNT(DISTINCT "Order ID") AS Total_Orders
FROM sales_data
GROUP BY Month
ORDER BY Month;
"""

# Execute the SQL query
result = pd.read_sql_query(query, conn)

# Display monthly order counts
result

,Month,Total_Orders
0,1,9293
1,2,11496
2,3,14549
3,4,17528
4,5,15836
5,6,12989
6,7,13761
7,8,11484
8,9,11202
9,10,19436


In [16]:
#  Compare products based on quantity sold and revenue generated

query = """
SELECT
    Product,
    SUM("Quantity Ordered") AS Total_Quantity,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY Product
ORDER BY Total_Revenue DESC;
"""

# Execute the SQL query
result = pd.read_sql_query(query, conn)

# Display the top 10 products by revenue
result.head(10)

,Product,Total_Quantity,Total_Revenue
0,Macbook Pro Laptop,4725,8032500.00
1,iPhone,6847,4792900.00
2,ThinkPad Laptop,4128,4127958.72
3,Google Phone,5529,3317400.00
4,27in 4K Gaming Monitor,6239,2433147.61
5,34in Ultrawide Monitor,6192,2352898.08
6,Apple Airpods Headphones,15637,2345550.00
7,Flatscreen TV,4813,1443900.00
8,Bose SoundSport Headphones,13430,1342865.70
9,27in FHD Monitor,7541,1131074.59


In [18]:
# STEP 6: Create the final Excel workbook

# Monthly Revenue
monthly_revenue = pd.read_sql_query("""
SELECT
    Month,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY Month
ORDER BY Month;
""", conn)

# Product Performance
product_performance = pd.read_sql_query("""
SELECT
    Product,
    SUM("Quantity Ordered") AS Total_Quantity,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY Product
ORDER BY Total_Revenue DESC;
""", conn)

# City Revenue
city_revenue = pd.read_sql_query("""
SELECT
    City,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY City
ORDER BY Total_Revenue DESC;
""", conn)

# State Revenue
state_revenue = pd.read_sql_query("""
SELECT
    State,
    ROUND(SUM(Sales), 2) AS Total_Revenue
FROM sales_data
GROUP BY State
ORDER BY Total_Revenue DESC;
""", conn)

# Monthly Orders
monthly_orders = pd.read_sql_query("""
SELECT
    Month,
    COUNT(DISTINCT "Order ID") AS Total_Orders
FROM sales_data
GROUP BY Month
ORDER BY Month;
""", conn)


# Create one Excel workbook with multiple sheets
with pd.ExcelWriter("ecommerce_sales_analysis.xlsx", engine="openpyxl") as writer:

    df.to_excel(writer, sheet_name="Cleaned Data", index=False)
    monthly_revenue.to_excel(writer, sheet_name="Monthly Revenue", index=False)
    product_performance.to_excel(writer, sheet_name="Product Performance", index=False)
    city_revenue.to_excel(writer, sheet_name="City Revenue", index=False)
    state_revenue.to_excel(writer, sheet_name="State Revenue", index=False)
    monthly_orders.to_excel(writer, sheet_name="Monthly Orders", index=False)

print("Excel file created successfully!")

Excel file created successfully!


In [19]:
# Download the Excel workbook

from google.colab import files

files.download("ecommerce_sales_analysis.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>